# `lidar-rs` Surface Inundation Dynamics Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/lidar-rs/blob/main/examples/inundation_demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/lidar-rs.svg)](https://pypi.org/project/lidar-rs/)
[![crates.io](https://img.shields.io/crates/v/lidar.svg)](https://crates.io/crates/lidar)

End-to-end pure-Rust port of the 7-step ArcGIS Hydrology Analyst inundation workflow from [`opengeos/lidar`](https://github.com/opengeos/lidar) — running in Google Colab with **zero ArcGIS, ArcPy, or GDAL dependencies**:

1. Smooth DEM (`MedianFilter`) and extract sinks (`extract_sinks`)
2. Delineate contributing sink catchments (`delineate_catchments`)
3. Trace overland spill flow paths between sinks (`delineate_flow_paths`)
4. Delineate nested depression hierarchy with per-level shapefiles (`delineate_depressions`)
5. Delineate nested catchment hierarchy & zonal DEM stats (`delineate_catchment_hierarchy`)
6. Simulate time-series surface inundation dynamics (`simulate_inundation`)
7. Render & display the shaded-relief inundation GIF animation (`PlayAnimation`)

In [ ]:
%pip install -q lidar-rs matplotlib pandas tifffile

In [ ]:
import logging
import urllib.request
from pathlib import Path

import lidar_rs as lidar
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile

logging.getLogger("tifffile").setLevel(logging.ERROR)
print(f"lidar-rs version: {lidar.__version__}")

In [ ]:
base_url = "https://raw.githubusercontent.com/opengeos/lidar-rs/main/test_data"
out_dir = Path("inundation_output")
out_dir.mkdir(exist_ok=True)

dem_path = out_dir / "raw_dem.tif"
if not dem_path.exists():
    urllib.request.urlretrieve(f"{base_url}/dem.tif", dem_path)

# 1. Smooth DEM & extract sinks
med_path = out_dir / "median.tif"
lidar.MedianFilter(dem_path, kernel_size=3, out_file=med_path)
sinks = lidar.extract_sinks(med_path, min_size=1000, out_dir=out_dir)

# 2. Delineate catchments
cats = lidar.delineate_catchments(
    sinks["refined_dem_path"], sinks["region_path"], out_dir / "catchments"
)
print("Catchments:")
display(pd.DataFrame(cats["records"]))

# 3. Delineate overland flow paths
flowpaths = lidar.delineate_flow_paths(
    sinks["filled_dem_path"],
    sinks["region_path"],
    out_dir / "flowpath.shp",
    rain_intensity_cm_h=5.0,
    refined_dem=sinks["refined_dem_path"],
)
print(f"Traced {len(flowpaths['records'])} overland flow path(s)")

# 4. Delineate nested depression hierarchy
deps = lidar.delineate_depressions(
    sinks["sink_path"],
    min_size=1000,
    min_depth=0.3,
    interval=0.3,
    out_dir=out_dir,
    bool_level_shp=True,
)
print(f"Nested depressions: {len(deps['depressions'])} (max level = {deps['max_level']})")

# 5. Delineate catchment hierarchy
hir_tif = out_dir / "catchment_hir.tif"
hir = lidar.delineate_catchment_hierarchy(
    sinks["refined_dem_path"], out_dir / "shp-level", hir_tif
)
display(pd.DataFrame(hir["records"]).head(10))

In [ ]:
# 6. Simulate inundation dynamics (5 cm/h rainfall over 20 hours, 1h time step)
sim_dir = out_dir / "simulation"
sim = lidar.simulate_inundation(
    sinks["sink_path"],
    hir_tif,
    sim_dir,
    min_size=1000,
    min_depth=0.2,
    interval=0.2,
    rain_intensity_cm_h=5.0,
    rain_duration_h=20.0,
    time_step_h=1.0,
)

df_steps = pd.DataFrame(sim["steps"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(df_steps["time_hours"], df_steps["inundated_fraction"] * 100.0, marker="o")
axes[0].set_xlabel("Rainfall duration (hours)")
axes[0].set_ylabel("Inundated sink area (%)")
axes[0].set_title("Inundation Growth Curve (5 cm/h)")
axes[0].grid(True, alpha=0.3)

ft = tifffile.imread(sim["fill_time_path"])
im = axes[1].imshow(np.ma.masked_less(ft, 0), cmap="viridis_r")
axes[1].set_title("Time to Inundation (hours)")
axes[1].axis("off")
fig.colorbar(im, ax=axes[1], shrink=0.85, label="Hours")
plt.tight_layout()
plt.show()

In [ ]:
# 7. Render and display the shaded-relief inundation animation GIF
lidar.PlayAnimation(
    sinks["refined_dem_path"],
    sim_dir,
    out_dir / "inundation.gif",
    frame_delay_ms=150,
    ping_pong=True,
    time_step_h=1.0,
    rain_intensity_cm_h=5.0,
    display=True,
)